# 01. How `dict` and `set` work

**Tier:** Deep. **Why you'd use this:** Membership tests on a growing set of ids should stay fast, and you need to know why a list cannot be a key while a tuple of strings can.

**After this lesson you can:** explain insertion order; explain why a key's hash must stay put; choose a set when a list scan shows up in a timing.
About 30 minutes. **Uses:** [Dicts](../../course/04-collections/03-dicts-lesson.ipynb), [Sets](../../course/04-collections/04-sets-lesson.ipynb), [How list works](../03-lists/01-dynamic-array-lesson.ipynb)


## Quick review

1. Why is `append` amortized cheap?
2. Why is `insert(0, x)` expensive on a long list?

<details><summary>Answers</summary>

Spare capacity means most appends do not move the whole array. Inserting at the front shifts every later item.
</details>


## The idea

A dict maps a key to a value. A set stores keys without values. Both are hash tables. The key's hash is an integer. That integer picks a slot in an array. If the slot is empty, the entry goes there. If another key already sits there, the table looks at the next slots until it finds room or finds an equal key.

Equal keys must have equal hashes, or the table looks in the wrong place and misses a key that is sitting in the array. A hash that changes after insertion is the same failure: the entry stays where the old hash put it, and the new hash looks elsewhere.

Insertion order is part of the table. Updating a key does not move it. Iterating yields keys in the order they were first inserted.

<details><summary>Explain more</summary>

`hash(obj)` is that integer. Strings, numbers, and tuples of hashable objects have it. A list does not, because a list can change and a hash that changes cannot be a stable address. A set is a dict with no values, so the same rules apply.

CPython detail: the numeric value of `hash("a")` is salted per process so attackers cannot predict bucket collisions. Compare hashes for equality inside one process. Do not store the number.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

This is `HashMap` / `HashSet`, with a required `hashCode`/`equals` contract. Python remembers insertion order. A list is not a legal key, the way a mutable object with a changing hash is not a legal key.
</details>


## See it work

A scoreboard keys players by an id that must not change while the player is in the table. An integer id is hashable and immutable, so the slot stays valid. A list of ids is not a key. A custom object is a key only while its hash and equality stay consistent with each other and stable for as long as it sits in the table.


In [1]:
class Player:
    def __init__(self, player_id: int) -> None:
        self.player_id = player_id

    def __hash__(self) -> int:
        return hash(self.player_id)

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Player) and self.player_id == other.player_id


def scoreboard(players: list[int]) -> dict[int, int]:
    # 1. The id is the key. The score starts at zero.
    board = {player_id: 0 for player_id in players}
    return board


board = scoreboard([4, 1, 4])
print(list(board))
print(board[4])
ada = Player(4)
by_player = {ada: 10}
print(by_player[Player(4)])

[4, 1]
0
10


| step | key identity | hash input | found? |
| --- | --- | --- | --- |
| insert Player(4) | object A | 4 | stored |
| lookup Player(4) | object B | 4, and B == A | yes, value 10 |

```
hash(4) --> slot --> (key A, 10)
lookup hashes 4, then == against the key in that slot
```

A second `Player(4)` is a different object and an equal key, so it finds the entry. Equality, not `is`, decides "same key" after the hash has picked a slot.


## Check yourself

Answer these from the example above before you run the cell.


1. Why does assigning `ranks["ada"] = 3` leave `ada` as the first key?
2. Why does `len(board)` stay 1 after `player_id` changes, if `ada in board` is false?
3. Why must equal players share a hash?

<details><summary>Answers</summary>

1. The key was already present. The assignment replaces the value in that entry. First-insertion order only records the key the first time it appears.
2. Nothing deleted the entry. Lookup uses the new hash and does not walk every slot, so it does not stumble onto the old one. The array still holds the pair.
3. The hash chooses the slot. If two equal players hashed differently, the lookup for one would never compare itself to the other, and the table would store both or miss one.
</details>


In [2]:
ranks = {}
ranks["ada"] = 1
ranks["bea"] = 2
ranks["ada"] = 3
print(list(ranks))
print(ranks["ada"])

['ada', 'bea']
3


<details><summary>Check your answers</summary>

- The keys are `['ada', 'bea']`. Assigning `ada` again replaced the value and left the key where it was first inserted. Order is first-insertion order, not last-update order, and it is not arbitrary.
- `ranks["ada"]` is `3`, the new value.
</details>


## Why it works this way

The table is an array of entries, larger than the number of keys, so many slots are empty. The hash picks a starting index. If that slot holds a different key, the lookup walks forward — a collision. With enough empty slots the walk is short, so a typical lookup looks at a handful of slots. The worst case, every key colliding into one chain, is a scan. Python keeps the table from getting too full so the average stays short. Growing copies entries into a bigger array, paid rarely.

The contract has two parts. If `a == b`, then `hash(a) == hash(b)`. Otherwise equal keys land in different slots and the table never compares them. And the hash of a key must not change while the key is in the table. The slot was chosen with the old hash; a lookup uses the new hash and starts somewhere else. The entry is still in the array — `len` does not drop — but lookup returns nothing. That is a lost entry, not a deletion.

Defining `__eq__` without `__hash__` makes the class unhashable. Python sets `__hash__` to `None` so you cannot put an object into a set whose equality changed while its default identity-hash stayed the same. The default hash is based on identity. If equality is based on a field, the default hash would break the contract whenever two equal objects are not the same object.

A set is this table without the value half. `x in some_set` is the lookup above; `x in some_list` walks every element. The timing cell shows that gap on a few thousand items. Insertion order on a dict is a side array of indices — a language guarantee, not a CPython accident.

CPython detail: string and bytes hashes are salted with a random seed at startup. `hash("a")` in two processes need not match. What the lesson prints is whether two equal keys share a hash, which is the contract.

The design reason is the trade. A scan is simple and uses no extra slots. A hash table spends memory and a careful contract to avoid the scan. Use it when you look keys up. Do not use it when you need to hash a value that is going to change.


In [3]:
left = Player(4)
right = Player(4)
print(left == right, hash(left) == hash(right), left is right)
board = {left: 10}
print(right in board, len(board))
left.player_id = 5
print(left in board, right in board, len(board))

True True False
True 1
False False 1


In [4]:
import timeit

items = list(range(4000))
keys = set(items)
needle = 3999
print(round(timeit.timeit(lambda: needle in items, number=2000), 4))
print(round(timeit.timeit(lambda: needle in keys, number=2000), 4))

0.0556
0.0001


## When to use it

Use a dict when you have a key and want a value, and a set when you only need "have I seen this". Use a list when you need order, duplicates, or indexing by position. A list of pairs is fine for a record you will scan once — wrong for a thousand membership tests.

Keys should be immutable: numbers, strings, tuples of those. A tuple that contains a list is not hashable. Do not use a custom object as a key unless equality and hash are based on fields you will not assign while it is in the table.

If you need order and fast lookup, a dict already keeps insertion order. You do not need a list beside it unless the order is something other than first insertion.


## Common mistake

Changing a field that `__hash__` reads, after the object is a key, leaves the entry in the table and makes the lookup miss. `len` stays 1. Nothing raises. The key is not gone. The address is stale.


In [5]:
ada = Player(4)
board = {ada: 10}
ada.player_id = 5
print(ada in board)
print(len(board))
print(list(board.values()))

False
1
[10]


The prints are `False`, `1`, and `[10]`. The value is still stored. Lookup starts at the hash of 5 and never compares the key that was stored under the hash of 4. The fix does not mutate the id. Look the player up with a new `Player(4)`, or store the score under the int `4` and leave the object out of the key.


In [6]:
ada = Player(4)
board = {ada: 10}
print(Player(4) in board)
print(board[Player(4)])

True
10


A list as a key fails immediately. The last line of the error is `TypeError`: the list is unhashable. That refusal is the contract enforced up front, instead of a lost entry later.


In [7]:
seen = {}
seen[[1, 2]] = "row"

TypeError: cannot use 'list' as a dict key (unhashable type: 'list')

## Try it

Store the score under `ada.player_id` instead of under `ada`, then change `ada.player_id` and look up `4`. Why does that lookup still find the score, and which object is no longer acting as the key?


## What this explains

- [Dicts](../../course/04-collections/03-dicts-lesson.ipynb): lookup by key, and why the key has to be hashable.
- [Sets](../../course/04-collections/04-sets-lesson.ipynb): membership without a scan, same table, no values.
- [How list works](../03-lists/01-dynamic-array-lesson.ipynb): `in` on a list walks pointers. The hash table is the alternative that spends empty slots to avoid that walk.


## Recap

- A hash picks a slot. Equality confirms the key. Equal keys must hash equal, and the hash must not change while the key is stored.
- Insertion order is first insertion. An update does not move the key.
- A set is this table without values. A list as a key is refused because its hash could not stay stable.

Next: [Numbers and strings](../05-numbers/01-numbers-strings-lesson.ipynb).
